# Data Cleaning — E-commerce Customer Purchase Behavior Analysis

## Objective

This notebook prepares the raw e-commerce transaction dataset for analysis by identifying and handling invalid transactions, missing values, duplicate records, and operational records that are not relevant to retail product sales.

The cleaned dataset will be used for transaction-level, customer-level, product-level, sales, and geographic analysis.

In [85]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Load Dataset

The dataset contains historical e-commerce transaction records.

The first step is to load the raw dataset and inspect its structure before performing any data cleaning.

In [86]:
#membaca csv
df = pd.read_csv("raw Dataset (real).csv")


In [87]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34


## 2. Initial Data Understanding

The raw dataset contains 541,909 transaction records and 9 variables.

The initial inspection focuses on:
- Dataset structure and data types
- Missing values
- Invalid transaction values
- Duplicate records
- Operational or non-product transaction records

In [88]:
df.shape

(541909, 9)

In [89]:
df.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country', 'Revenue'],
      dtype='object')

In [90]:
df.dtypes

InvoiceNo       object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
UnitPrice      float64
CustomerID     float64
Country         object
Revenue        float64
dtype: object

### Data Type Conversion

The `InvoiceDate` column is converted from an object type to datetime so that it can be properly used for time-based analysis in the EDA stage.

In [91]:
df["InvoiceDate"]= pd.to_datetime(df["InvoiceDate"])

In [92]:
df.dtypes

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
Revenue               float64
dtype: object

## 3. Missing Value Investigation

Missing values were assessed to determine whether they should be removed, retained, or handled based on their analytical relevance.

CustomerID missing values were retained because these records still contain valid transaction, product, revenue, date, and geographic information. However, customer-level analysis will only use records with identified CustomerID.

In [93]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
Revenue             0
dtype: int64

A total of 1,454 records have missing Description values.

### Investigating Missing `Description`

The missing values in `Description` are investigated further before deciding whether they should be removed.

The purpose is to determine whether these records represent meaningful transactions or potentially problematic records.

In [94]:
#mengolah missing value 
#1. description 
df[df["Description"].isnull()].head(10)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom,0.0
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom,0.0
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom,0.0
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom,0.0
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom,0.0
1988,536550,85044,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom,0.0
2024,536552,20950,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom,0.0
2025,536553,37461,NaN,3,2010-12-01 14:35:00,0.0,NaN,United Kingdom,0.0
2026,536554,84670,NaN,23,2010-12-01 14:35:00,0.0,NaN,United Kingdom,0.0
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom,0.0


In [95]:
df[df["Description"].isnull()]["StockCode"].value_counts().head(10)

StockCode
35965    10
23084    10
22084     9
22451     6
22501     5
21784     5
21033     5
21067     5
23348     5
22866     5
Name: count, dtype: int64

In [96]:
df[df["StockCode"] == "35965"][["StockCode","Description"]]

,StockCode,Description
2889,35965,FOLKART HEART NAPKIN RINGS
6017,35965,FOLKART HEART NAPKIN RINGS
7205,35965,NaN
8071,35965,FOLKART HEART NAPKIN RINGS
10678,35965,FOLKART HEART NAPKIN RINGS
...,...,...
347758,35965,NaN
349563,35965,FOLKART HEART NAPKIN RINGS
454169,35965,NaN
464522,35965,NaN


In [97]:
df[df["Description"].isnull()][
    ["InvoiceNo","StockCode","Description","Quantity","CustomerID","Revenue", "UnitPrice"]
].head(20)

,InvoiceNo,StockCode,Description,Quantity,CustomerID,Revenue,UnitPrice
622,536414,22139,NaN,56,NaN,0.0,0.0
1970,536545,21134,NaN,1,NaN,0.0,0.0
1971,536546,22145,NaN,1,NaN,0.0,0.0
1972,536547,37509,NaN,1,NaN,0.0,0.0
1987,536549,85226A,NaN,1,NaN,0.0,0.0
1988,536550,85044,NaN,1,NaN,0.0,0.0
2024,536552,20950,NaN,1,NaN,0.0,0.0
2025,536553,37461,NaN,3,NaN,0.0,0.0
2026,536554,84670,NaN,23,NaN,0.0,0.0
2406,536589,21777,NaN,-10,NaN,0.0,0.0


In [98]:
df[df["UnitPrice"]==0]["InvoiceNo"].head(20)

622     536414
1970    536545
1971    536546
1972    536547
1987    536549
1988    536550
2024    536552
2025    536553
2026    536554
2406    536589
4347    536764
4348    536765
6391    536941
6392    536942
7187    536995
7188    536996
7189    536997
7190    536998
7191    536999
7192    537000
Name: InvoiceNo, dtype: object

In [99]:
df[df["UnitPrice"] == 0]["InvoiceNo"].astype(str).str.startswith("C").sum()

0

### Invalid Unit Price

Records with UnitPrice = 0 do not represent valid revenue-generating product sales.

Therefore, records with UnitPrice <= 0 are excluded from the analytical dataset.

In [100]:
#untuk compare missing value both of them 
print((df["UnitPrice"] == 0).sum())

df["Description"].isnull().sum()

2515


1454

In [101]:
#check apakah semua unit price yang 0, memiliki description yang NULL?
(
    df[df["Description"].isnull()]["UnitPrice"]==0
).all()

True

In [102]:
df[df["Quantity"]<=0].head(30)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom,-27.50
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom,-4.65
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom,-19.80
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom,-6.96
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom,-6.96
238,C536391,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom,-6.96
239,C536391,21484,CHICK GREY HOT WATER BOTTLE,-12,2010-12-01 10:24:00,3.45,17548.0,United Kingdom,-41.40
240,C536391,22557,PLASTERS IN TIN VINTAGE PAISLEY,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom,-19.80
241,C536391,22553,PLASTERS IN TIN SKULLS,-24,2010-12-01 10:24:00,1.65,17548.0,United Kingdom,-39.60
939,C536506,22960,JAM MAKING SET WITH JARS,-6,2010-12-01 12:38:00,4.25,17897.0,United Kingdom,-25.50


### Cancelled Transactions

Transactions with InvoiceNo beginning with "C" are treated as cancelled transactions and are excluded from the sales analysis.

In [103]:
(df[(df["Quantity"] <= 0) & (df["InvoiceNo"].str.startswith("C"))]).shape[0]

9288

In [104]:
df[(df["Description"].isnull())]["CustomerID"].head(20)

622    NaN
1970   NaN
1971   NaN
1972   NaN
1987   NaN
1988   NaN
2024   NaN
2025   NaN
2026   NaN
2406   NaN
4347   NaN
4348   NaN
7187   NaN
7188   NaN
7189   NaN
7190   NaN
7191   NaN
7192   NaN
7193   NaN
7194   NaN
Name: CustomerID, dtype: float64

In [105]:
#apakah setiap kali kolom description kosong, kolom customerID juga kosong
(
    df[df["Description"].isnull()]["CustomerID"].isnull()
).all()

True

In [106]:
df[df["UnitPrice"] == 0]["CustomerID"].isna().all()

False

In [107]:
original_rows = len(df)

print(f"Rows before cleaning : {original_rows}")


Rows before cleaning : 541909


In [108]:
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]
df = df[df["UnitPrice"] > 0]

print(f"Rows after cleaning  : {len(df)}")
print(f"Rows removed         : {original_rows - len(df)}")

Rows after cleaning  : 530104
Rows removed         : 11805


## 4. Duplicate Records

Duplicate records were identified and removed to prevent repeated transaction lines from affecting revenue, quantity, and transaction-level analysis.

In [109]:
df.duplicated().sum()

5226

In [110]:
df = df.drop_duplicates()

In [111]:
df.shape

(524878, 9)

In [112]:
df.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID,Revenue
count,524878.000000,524878,524878.000000,392692.000000,524878.000000
mean,10.616600,2011-07-04 15:30:16.317049088,3.922573,15287.843865,20.275399
min,1.000000,2010-12-01 08:26:00,0.001000,12346.000000,0.001000
25%,1.000000,2011-03-28 12:13:00,1.250000,13955.000000,3.900000
50%,4.000000,2011-07-20 11:22:00,2.080000,15150.000000,9.920000
75%,11.000000,2011-10-19 11:41:00,4.130000,16791.000000,17.700000
max,80995.000000,2011-12-09 12:50:00,13541.330000,18287.000000,168469.600000
std,156.280031,NaN,36.093028,1713.539549,271.693566


In [113]:
df.boxplot(column="Quantity")

<Axes: >

## 5. Operational and Non-Product Records

Records representing operational or accounting activities rather than retail product sales are excluded from the physical product analysis.

The following records were removed:
- MANUAL
- ADJUST BAD DEBT
- AMAZON FEE

A total of 320 records were removed.

In [ ]:
#change all text in description column into uppercase 
df = df[df["Description"].str.upper() != "MANUAL"]

In [115]:
df = df[df["Description"].str.upper() != "ADJUST BAD DEBT"]

In [116]:
df = df[df["Description"].str.upper() != "AMAZON FEE"]

In [117]:
df.shape

(524558, 9)

## 6. Final Data Validation

After cleaning, the dataset is validated to ensure that invalid and duplicate records have been successfully removed.

In [124]:
#FINAL VALIDATE DATA
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 524558 entries, 0 to 541908
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    524558 non-null  object        
 1   StockCode    524558 non-null  object        
 2   Description  524558 non-null  object        
 3   Quantity     524558 non-null  int64         
 4   InvoiceDate  524558 non-null  datetime64[ns]
 5   UnitPrice    524558 non-null  float64       
 6   CustomerID   392413 non-null  float64       
 7   Country      524558 non-null  object        
 8   Revenue      524558 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(4)
memory usage: 40.0+ MB


In [119]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     132145
Country             0
Revenue             0
dtype: int64

In [120]:
df.duplicated().sum()

0

In [121]:
df.dtypes

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
Revenue               float64
dtype: object

In [122]:
df.shape

(524558, 9)

In [125]:
print("Cancelled transactions:", 
      df["InvoiceNo"].astype(str).str.startswith("C").sum())

print("UnitPrice <= 0:", 
      (df["UnitPrice"] <= 0).sum())

print("Quantity <= 0:", 
      (df["Quantity"] <= 0).sum())

print("Missing Description:", 
      df["Description"].isnull().sum())

print("Duplicate rows:", 
      df.duplicated().sum())

Cancelled transactions: 0
UnitPrice <= 0: 0
Quantity <= 0: 0
Missing Description: 0
Duplicate rows: 0


### Validation Result

The final dataset contains **524,558 records and 9 variables**.

Validation confirms that:
- No cancelled transactions remain.
- No records with non-positive UnitPrice remain.
- No records with non-positive Quantity remain.
- No missing Description values remain.
- No duplicate records remain.

CustomerID missing values were intentionally retained because these records remain useful for transaction-level, product, sales, and geographic analysis. Customer-level analysis is restricted to records with identified CustomerID.

## 10. Export Cleaned Dataset

The cleaned dataset is exported for use in the exploratory data analysis stage.

In [123]:
df.to_csv("cleaned_data.csv", index=False)